In [35]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean

load_dotenv()

auth_token = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(
    auth_token=auth_token,
    base_url="https://flow.ciandt.com/flow-llm-proxy/"
)

model = "bedrock/anthropic.claude-4-5-haiku"

In [36]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [56]:
import json
import re


def extract_json_array(text):
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON array found in response:\n{text}")
    return json.loads(match.group())


def extract_json_object(text):
    decoder = json.JSONDecoder()
    start = text.index('{')
    obj, _ = decoder.raw_decode(text, start)
    return obj


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects. Example format:
[
    {
        "task": "Description of task",
        "format": "python" or "regex" or "json",
        "solution_criteria": some requirements to aim a better solution for the current task. it should not be an array, a single string.
    }
]
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    return extract_json_array(text)


In [57]:
dataset = generate_dataset()

with open("database.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [60]:
def run_prompt(test_case):
    """"Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task
<task>
{test_case["task"]}
<task>

* Consider the following solution criteria
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
* Example of response:
    ```code
    // solution content
    ```
* Do not add anything before "```code" or "```"
* Valid options for code are *ONLY* regex, json or python
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [61]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: 
<task>
{test_case["task"]}
</task>

Solution criteria:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Solution:
<solution>
{output}
</solution>

Provide your evaluation based on the Solution criteria as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages)
    print("solution", eval_text)
    return extract_json_object(eval_text)


In [51]:
import re
import ast

def validate_json(text):
    try:
        parsed_text =  text.replace("```json", "").replace("```", "").strip()
        print("As json:\n", parsed_text)
        json.loads(parsed_text)
        return 10
    except json.JSONDecodeError:
        return 0
    

def validate_python(text):
    try:
        parsed_text =  text.replace("```python", "").replace("```", "").strip()
        print("Script as python:\n", parsed_text)
        ast.parse(parsed_text)
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        parsed_text = text.replace("```regex", "").replace("```", "").strip()
        print("As regex:\n", parsed_text)
        re.compile(parsed_text)
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]

    if format == "json":
        return validate_json(response)
    elif format == "regex":
        return validate_regex(response)
    elif format == "python":
        return validate_python(response)
    else:
        return 0

In [42]:
def run_test_case(test_case): 
    """"Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)

    reasoning = model_grade["reasoning"]
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return { 
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [43]:
def run_eval(dataset):
    """"Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [62]:
with open("database.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

solution ```json
{
  "strengths": [
    "Comprehensive input validation with type checking and length constraints properly enforced",
    "Clear docstring explaining all AWS S3 naming rules with good code readability",
    "Handles most edge cases including consecutive hyphens, leading/trailing hyphens, and uppercase letters through regex and explicit checks"
  ],
  "weaknesses": [
    "Regex pattern has a subtle flaw: the optional group `([a-z0-9\\-]*[a-z0-9])?` allows single-character bucket names (e.g., 'a' matches), violating the 3-character minimum despite the length check providing runtime protection",
    "Consecutive hyphen check `'--' in bucket_name` is redundant since the regex pattern `[a-z0-9\\-]*[a-z0-9]` already prevents consecutive hyphens from matching; this adds unnecessary overhead",
    "Missing validation for AWS's additional rule that bucket names cannot be formatted as IP addresses (e.g., '192.168.1.1' should be invalid)"
  ],
  "reasoning": "The solution correctl

In [ ]:
print(json.dumps(results, indent=2))

[{"output": "```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name):\n    \"\"\"\n    Validates if a given string is a valid AWS S3 bucket name.\n    \n    Rules:\n    - Length: 3-63 characters\n    - Characters: lowercase letters, numbers, hyphens only\n    - Must start and end with letter or number\n    - No consecutive hyphens\n    \"\"\"\n    if not isinstance(bucket_name, str):\n        return False\n    \n    if len(bucket_name) < 3 or len(bucket_name) > 63:\n        return False\n    \n    if not re.match(r'^[a-z0-9]([a-z0-9\\-]*[a-z0-9])?$', bucket_name):\n        return False\n    \n    if '--' in bucket_name:\n        return False\n    \n    return True\n```", "test_case": {"task": "Create a Python function that validates if a given string is a valid AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, hyphens only, must start and end with letter or number)", "format": "python", "solution_criteria": "The function should retu